# Mapillary-Abdeckung — Osnabrück

Erfasst alle Mapillary-Bildstandorte im Stadtgebiet und wertet die Abdeckung aus.

**Warum Vector Tiles und nicht die Bbox-Suche?**
`graph.mapillary.com/images?bbox=` ist eine Suchschnittstelle und antwortet
nachweislich unvollständig — gemessen wurde eine Kachel mit 30 Treffern, die
beim Vierteln 39 ergab. Es gibt keine Kachelgröße und keine Trefferzahl, an der
man das erkennen könnte, und es kommt weder Fehler noch Warnung.
Vector Tiles sind dieselbe Quelle, aus der mapillary.com seine Karte zeichnet:
vollständig per Konstruktion, und für Osnabrück reichen ~130 Abfragen.


`.env` im Projektordner mit `MAPILLARY_TOKEN=MLY|...` (Vorlage: `.env.example`).
Alle übrigen Parameter stehen in `config.yaml`.


## 1. Konfiguration


In [ ]:
import json, math, os, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import mapbox_vector_tile
import numpy as np
import pandas as pd
import requests
import yaml
from requests.adapters import HTTPAdapter, Retry

import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
from shapely.geometry import Point


def find_upwards(name):
    # Erlaubt, das Notebook aus eval/ oder aus dem Projektwurzelverzeichnis
    # zu starten, ohne Pfade anzupassen.
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None


# Parameter: versioniert, damit alle im Team denselben Datensatz erzeugen.
cfg_file = find_upwards("config.yaml")
assert cfg_file, "config.yaml nicht gefunden (liegt im Projektwurzelverzeichnis)."
CFG = yaml.safe_load(cfg_file.read_text())

CITY_NAME   = CFG["city"]
ZOOM        = CFG["zoom"]
MAX_WORKERS = CFG["max_workers"]

# Token: geheim, deshalb bewusst nicht in config.yaml.
if env_file := find_upwards(".env"):
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip("'\""))

TOKEN = os.environ.get("MAPILLARY_TOKEN", "")
assert TOKEN.startswith("MLY|"), (
    "Kein Mapillary-Token. Datei .env anlegen:\n    MAPILLARY_TOKEN=MLY|dein|token"
)

DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
F_TILES = DATA_DIR / "vt_tiles.json"     # Rohpunkte je Kachel, dient als Checkpoint
F_ALL   = DATA_DIR / "vt_images.csv"     # alle Punkte im Kachelbereich
F_CITY  = DATA_DIR / "images_city.csv"   # Endergebnis: nur innerhalb der Stadtgrenze

ox.settings.use_cache = True
plt.rcParams["figure.dpi"] = 120

print(f"{CITY_NAME} | Zoom {ZOOM} | Konfig: {cfg_file.name} | Token ✓")


## 2. Hilfsfunktionen

Kachelmathematik nach der üblichen Web-Mercator-Konvention (wie OSM und Google
Maps), Dekodierung der binären Kachelformate und eine HTTP-Sitzung mit Retry.


In [ ]:
TILE_URL = "https://tiles.mapillary.com/maps/vtp/mly1_public/2/{z}/{x}/{y}"

# Die Dekodierbibliothek legt den Ursprung unten links, die Kachelspezifikation
# oben links. Gemessen gegen bekannte Koordinaten aus der Graph-API:
#   y_down=False -> 0,3 m Abweichung      y_down=True -> 699 m
# Rät man hier falsch, liegt alles an der Kachelmitte gespiegelt — und die Karte
# sieht trotzdem plausibel aus. Prüfzelle dazu im Anhang.
Y_DOWN = False

# Nur diese Felder liefern die Kacheln; sie reichen für die Abdeckungsanalyse.
# Kameratyp (Panorama, Fisheye) steht dort NICHT drin und muss später über die
# Graph-API geholt werden — für die paar tausend Bilder des Trainingssets.
PROPS = {"id": "image_id", "sequence_id": "sequence_id",
         "captured_at": "captured_at", "compass_angle": "compass_angle"}


def make_session():
    s = requests.Session()
    s.mount("https://", HTTPAdapter(
        max_retries=Retry(total=5, backoff_factor=0.6,
                          status_forcelist=[429, 500, 502, 503, 504],
                          allowed_methods=["GET"]),
        pool_maxsize=MAX_WORKERS * 4))
    return s


def deg2tile(lon, lat, z):
    n = 2 ** z
    return (int((lon + 180) / 360 * n),
            int((1 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2 * n))


def px2deg(tx, ty, px, py, extent, z):
    # Pixel innerhalb einer Kachel -> Lon/Lat
    n = 2 ** z
    if not Y_DOWN:
        py = extent - py
    wx, wy = (tx + px / extent) / n, (ty + py / extent) / n
    return (wx * 360 - 180,
            math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * wy)))))


def load_tile(session, tx, ty):
    r = session.get(TILE_URL.format(z=ZOOM, x=tx, y=ty),
                    params={"access_token": TOKEN}, timeout=60)
    r.raise_for_status()
    layer = mapbox_vector_tile.decode(r.content).get("image")
    if layer is None:
        return []
    extent = layer.get("extent", 4096)
    out = []
    for f in layer.get("features", []):
        geom = f.get("geometry") or {}
        if geom.get("type") != "Point":
            continue
        lon, lat = px2deg(tx, ty, *geom["coordinates"], extent, ZOOM)
        row = {"lon": lon, "lat": lat}
        row.update({PROPS[k]: v for k, v in (f.get("properties") or {}).items()
                    if k in PROPS})
        out.append(row)
    return out


## 3. Stadtgebiet und Kachelbereich


In [ ]:
city_polygon = ox.geocode_to_gdf(CITY_NAME).geometry.iloc[0]
UTM_CRS   = gpd.GeoSeries([city_polygon], crs="EPSG:4326").estimate_utm_crs()
city_area = gpd.GeoSeries([city_polygon], crs="EPSG:4326").to_crs(UTM_CRS).area.iloc[0] / 1e6

lon_min, lat_min, lon_max, lat_max = city_polygon.bounds
x0, y0 = deg2tile(lon_min, lat_max, ZOOM)     # obere linke Kachel
x1, y1 = deg2tile(lon_max, lat_min, ZOOM)     # untere rechte
TILES = [(x, y) for x in range(x0, x1 + 1) for y in range(y0, y1 + 1)]

print(f"Fläche: {city_area:.1f} km²   Kacheln: {len(TILES)}")


## 4. Kacheln laden

Der Checkpoint wird alle zehn Kacheln geschrieben. Ein Abbruch kostet damit
höchstens ein paar Sekunden Arbeit; beim erneuten Ausführen wird nur geladen,
was noch fehlt.


In [ ]:
done = json.loads(F_TILES.read_text()) if F_TILES.exists() else {}
todo = [t for t in TILES if f"{t[0]}_{t[1]}" not in done]
print(f"{len(todo)} von {len(TILES)} Kacheln offen")

if todo:
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        jobs = {pool.submit(lambda t: (t, load_tile(make_session(), *t)), t): t
                for t in todo}
        for i, job in enumerate(as_completed(jobs), 1):
            try:
                (tx, ty), pts = job.result()
                done[f"{tx}_{ty}"] = pts
            except Exception as e:
                print(f"\n  Kachel {jobs[job]} fehlgeschlagen: {type(e).__name__}: {e}")
            if i % 10 == 0 or i == len(todo):
                F_TILES.write_text(json.dumps(done))
                print(f"  {i}/{len(todo)} Kacheln — "
                      f"{sum(len(v) for v in done.values()):,} Punkte "
                      f"({time.time() - t0:.0f}s)", end="\r")
    F_TILES.write_text(json.dumps(done))

images = (pd.DataFrame([r for v in done.values() for r in v])
            .drop_duplicates(subset="image_id"))
images.to_csv(F_ALL, index=False)
print(f"\n\n{len(images):,} Punkte, {images['sequence_id'].nunique():,} Sequenzen "
      f"-> {F_ALL}")


## 5. Auf das Stadtgebiet schneiden


In [ ]:
images_gdf = gpd.GeoDataFrame(
    images, geometry=gpd.points_from_xy(images["lon"], images["lat"]), crs="EPSG:4326")
images_gdf = images_gdf[images_gdf.geometry.within(city_polygon)].reset_index(drop=True)
images_gdf.drop(columns="geometry").to_csv(F_CITY, index=False)

print(f"Im Stadtgebiet: {len(images_gdf):,} von {len(images):,} "
      f"({len(images_gdf) / len(images) * 100:.1f} %)")
print(f"Dichte: {len(images_gdf) / city_area:,.0f} Bilder/km²   -> {F_CITY}")


## 6. Verifikation gegen mapillary.com

Feste Orte statt „dichtester Bereich": Letzterer zeigt per Konstruktion die
beste Stelle und taugt deshalb nicht als Kontrolle. Die ausgegebene URL öffnet
denselben Ausschnitt auf mapillary.com — direkt nebeneinanderlegbar.


In [ ]:
SPOTS    = {k: tuple(v) for k, v in CFG["verification"]["spots"].items()}
RADIUS_M = CFG["verification"]["radius_m"]

for name, (clat, clon) in SPOTS.items():
    dlat = RADIUS_M / 111_320
    dlon = RADIUS_M / (111_320 * np.cos(np.radians(clat)))
    sel = images_gdf[images_gdf["lat"].between(clat - dlat, clat + dlat)
                     & images_gdf["lon"].between(clon - dlon, clon + dlon)]

    fig, ax = plt.subplots(figsize=(7.5, 7.5))
    try:
        street_net = ox.graph_from_point((clat, clon), dist=RADIUS_M, network_type="all")
        ox.plot_graph(street_net, ax=ax, node_size=0, edge_linewidth=0.6,
                      edge_color="#cccccc", bgcolor="white", show=False, close=False)
    except Exception as e:
        print(f"  (Straßennetz nicht geladen: {type(e).__name__})")

    ax.scatter(sel["lon"], sel["lat"], s=5, c="#2ca02c", alpha=0.5,
               edgecolors="none", zorder=5)
    ax.set_xlim(clon - dlon, clon + dlon)
    ax.set_ylim(clat - dlat, clat + dlat)
    ax.set_aspect(1 / np.cos(np.radians(clat)))    # Längengrade sind hier kürzer
    ax.set_title(f"{name} — {len(sel):,} Bilder im {RADIUS_M}-m-Umkreis")
    plt.tight_layout(); plt.show()

    print(f"{name:<15} {len(sel):>7,} Bilder, {sel['sequence_id'].nunique():>4} Sequenzen")
    print(f"   https://www.mapillary.com/app/?lat={clat}&lng={clon}&z=17\n")


## 7. Stadtweite Karte


In [ ]:
# network_type="drive" reicht als Hintergrund und lädt deutlich schneller
# als "all" (das zusätzlich Fuß- und Radwege enthält).
street_net = ox.graph_from_polygon(city_polygon, network_type="drive")

fig, ax = ox.plot_graph(street_net, node_size=0, edge_linewidth=0.35,
                        edge_color="#cccccc", bgcolor="white",
                        show=False, close=False, figsize=(14, 14))
ax.scatter(images_gdf["lon"], images_gdf["lat"], s=0.8, c="#2ca02c",
           alpha=0.35, edgecolors="none", zorder=5)
gpd.GeoSeries([city_polygon], crs="EPSG:4326").boundary.plot(
    ax=ax, color="black", linewidth=1.2, linestyle="--")
ax.set_title(f"{CITY_NAME.split(',')[0]} — {len(images_gdf):,} Mapillary-Bilder")
plt.tight_layout(); plt.savefig(DATA_DIR / "coverage_city.png"); plt.show()


## 8. Stadtteile bestimmen

Drei Filterstufen, jede gegen eine konkrete Fehlerquelle:

1. **Überlappung über 50 %** — wirft Nachbargemeinden raus, die nur eine Grenzlinie teilen
2. **Größenfilter** — wirft Mini-Artefakte und die Stadt selbst raus
3. **Hierarchie-Auflösung** — OSM führt oft Stadtteil *und* Ortsteil darin;
   ohne diesen Schritt werden dieselben Bilder doppelt gezählt


In [ ]:
DISTRICTS = CFG["districts"]
PLACE_TAGS        = DISTRICTS["place_tags"]
MIN_AREA_KM2       = DISTRICTS["min_area_km2"]
NESTING_THRESHOLD  = DISTRICTS["nesting_threshold"]
# Neu, mit Default -- muessen noch nicht in config.yaml stehen:
ADMIN_LEVELS       = DISTRICTS.get("admin_levels", ["9", "10"])
NESTING_STRATEGY   = DISTRICTS.get("nesting_strategy", "keep_coarser")  # oder "keep_finer"
MIN_DISTRICTS_REQ  = DISTRICTS.get("min_districts_required", 3)


def get_districts_by_place_tag(city_name, tags):
    try:
        gdf = ox.features_from_place(city_name, tags={"place": tags})
        gdf = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])]
        if "name" not in gdf.columns:
            return None
        gdf = gdf[gdf["name"].notna()][["name", "geometry"]].reset_index(drop=True)
        return gdf if len(gdf) > 0 else None
    except Exception as e:
        print(f"Strategie A (place-Tags) fehlgeschlagen: {e}")
        return None


def get_districts_by_admin_level(city_name, admin_levels):
    # admin_level wird NACH der Abfrage gefiltert, nicht als zweiter Tag-Key:
    # osmnx verknuepft mehrere Tag-Keys mit ODER, nicht UND -- sonst kaemen
    # Bundesland, Landkreis und Nachbargemeinden mit rein.
    try:
        gdf = ox.features_from_place(city_name, tags={"boundary": "administrative"})
        gdf = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])]
        if "name" not in gdf.columns or "admin_level" not in gdf.columns:
            return None
        gdf = gdf[gdf["admin_level"].isin(admin_levels)]
        gdf = gdf[gdf["name"].notna()][["name", "geometry"]].reset_index(drop=True)
        return gdf if len(gdf) > 0 else None
    except Exception as e:
        print(f"Strategie B (admin_level) fehlgeschlagen: {e}")
        return None


def diagnose_osm_districts(city_name):
    """Zeigt, welche Stadtteil-Schemata OSM fuer diese Stadt ueberhaupt kennt --
    hilft zu unterscheiden zwischen 'gibt es nicht' und 'Filter zu streng'."""
    print("\n" + "=" * 62)
    print(f"DIAGNOSE: Was kennt OSM fuer '{city_name}'?")
    print("=" * 62)
    for tag_key, tag_vals in [("place", ["suburb", "quarter", "neighbourhood",
                                         "borough", "village", "hamlet"]),
                              ("boundary", ["administrative"])]:
        try:
            gdf = ox.features_from_place(city_name, tags={tag_key: tag_vals})
            if len(gdf) == 0:
                print(f"\n{tag_key}: keine Treffer")
                continue
            print(f"\n{tag_key}: {len(gdf)} Objekte")
            for gtype, cnt in gdf.geometry.type.value_counts().items():
                mark = "  <- nutzbar" if gtype in ("Polygon", "MultiPolygon") else "  (keine Flaeche)"
                print(f"    {gtype:15s} {cnt:4d}{mark}")
            if tag_key == "place" and "place" in gdf.columns:
                print(f"    Werte: {dict(gdf['place'].value_counts())}")
            if tag_key == "boundary" and "admin_level" in gdf.columns:
                poly = gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])]
                if len(poly):
                    print(f"    admin_level (nur Flaechen): "
                          f"{dict(poly['admin_level'].value_counts().sort_index())}")
        except Exception as e:
            print(f"\n{tag_key}: Abfrage fehlgeschlagen ({e})")
    print("\n" + "-" * 62)
    print("Interpretation:")
    print("  - Nur 'Point' -> Stadtteile sind nur als Punkte getaggt, nicht als")
    print("    Flaechen. Eine flaechenbezogene Auswertung ist dann nicht moeglich.")
    print("  - Andere admin_level als oben -> ADMIN_LEVELS in config.yaml anpassen.")
    print("  - Keine Treffer -> Stadt ist in OSM nicht nach Stadtteilen gegliedert.")
    print("=" * 62 + "\n")


def resolve_nesting(gdf, strategy, threshold):
    """Entfernt verschachtelte Polygone (mehrere Hierarchie-Ebenen gleichzeitig
    getaggt, z.B. Stadtbezirk + darin liegender Stadtteil). Ohne diesen Schritt
    wuerden Bilder in zwei Ebenen gleichzeitig gezaehlt.

    Projiziert einmal auf das schon vorhandene UTM_CRS, danach laeuft die
    O(n^2)-Schleife auf shapely-.area statt auf wiederholten CRS-Transformationen.
    """
    gdf = gdf.sort_values("area_km2", ascending=False).reset_index(drop=True)
    if len(gdf) == 0:
        return gdf

    proj = gpd.GeoSeries(list(gdf.geometry.values), crs="EPSG:4326").to_crs(UTM_CRS)
    proj_geoms, proj_areas = list(proj.values), list(proj.area.values)

    to_drop, nested_pairs = set(), []
    for i in range(len(gdf)):
        if i in to_drop:
            continue
        for j in range(len(gdf)):
            if i == j or j in to_drop:
                continue
            if proj_areas[j] == 0 or proj_areas[i] <= proj_areas[j]:
                continue
            if not proj_geoms[i].intersects(proj_geoms[j]):
                continue
            inter = proj_geoms[j].intersection(proj_geoms[i])
            if inter.is_empty:
                continue
            if inter.area / proj_areas[j] > threshold:
                nested_pairs.append((gdf.loc[j, "name"], gdf.loc[i, "name"],
                                     inter.area / proj_areas[j]))
                to_drop.add(j if strategy == "keep_coarser" else i)
                if strategy == "keep_finer":
                    break

    if nested_pairs:
        print(f"\n{len(nested_pairs)} verschachtelte Stadtteil-Paare gefunden:")
        for inner, outer, frac in nested_pairs:
            print(f"  '{inner}' liegt zu {frac*100:.0f}% in '{outer}'")
        kept = "groessere" if strategy == "keep_coarser" else "kleinere"
        print(f"  -> Strategie '{strategy}': {kept} Einheit behalten\n")

    return gdf.drop(index=list(to_drop)).reset_index(drop=True)


# ---------------------------------------------------------------
print(f"Lade Stadtteile fuer: {CITY_NAME}")

print("\nStrategie A (place-Tags)...")
districts = get_districts_by_place_tag(CITY_NAME, PLACE_TAGS)
if districts is None or len(districts) < 2:
    print("Zu wenige Treffer, versuche Strategie B (admin_level)...")
    districts = get_districts_by_admin_level(CITY_NAME, ADMIN_LEVELS)

# HAS_DISTRICTS steuert, ob die stadtteilbezogenen Abschnitte laufen.
# Statt Ersatzeinheiten zu erfinden, die keine echten Stadtteile sind, werden
# diese Auswertungen sauber uebersprungen -- die stadtweite Analyse bleibt gueltig.
HAS_DISTRICTS = districts is not None and len(districts) > 0
districts_gdf = None

if not HAS_DISTRICTS:
    print("\n⚠️  Keine Stadtteil-Flaechen in OSM gefunden.")
    diagnose_osm_districts(CITY_NAME)
else:
    # Ungueltige Geometrien reparieren (selbstueberschneidende Polygone kommen
    # in OSM vor und wuerden intersection() spaeter mit TopologyException abbrechen).
    n_invalid = int((~districts.geometry.is_valid).sum())
    if n_invalid:
        print(f"{n_invalid} ungueltige Polygone repariert (buffer(0))")
        districts["geometry"] = districts.geometry.buffer(0)
    districts = districts[~districts.geometry.is_empty].reset_index(drop=True)

    districts["area_km2"] = (gpd.GeoSeries(list(districts.geometry.values), crs="EPSG:4326")
                             .to_crs(UTM_CRS).area.values / 1e6)

    # Ueberlappungs-Anteil mit der Stadtgrenze, vektorisiert, in UTM (nicht in
    # Grad -- sonst verzerrt die Breitengrad-abhaengige Flaechenverzerrung
    # das Verhaeltnis systematisch).
    proj_d = gpd.GeoSeries(list(districts.geometry.values), crs="EPSG:4326").to_crs(UTM_CRS)
    proj_city = gpd.GeoSeries([city_polygon], crs="EPSG:4326").to_crs(UTM_CRS).iloc[0]
    inter = np.array([g.intersection(proj_city).area if g.intersects(proj_city) else 0.0
                      for g in proj_d.values])
    own = proj_d.area.values
    districts["overlap_frac"] = np.divide(inter, own, out=np.zeros_like(inter), where=own > 0)

    n_start = len(districts)
    dbg = districts[["name", "area_km2", "overlap_frac"]].copy()

    # Obere Grenze bewusst bei der halben Stadtflaeche (nicht 0.9): ein
    # "Stadtteil", der fast die ganze Stadt abdeckt, ist meist ein falsch
    # eingeordnetes groesseres Verwaltungsgebiet, kein echter Stadtteil.
    districts = districts[
        (districts["overlap_frac"] > 0.5)
        & (districts["area_km2"] >= MIN_AREA_KM2)
        & (districts["area_km2"] < city_area * 0.5)
    ].reset_index(drop=True)

    districts = (districts.sort_values("area_km2", ascending=False)
                 .drop_duplicates(subset="name", keep="first").reset_index(drop=True))

    print(f"{n_start - len(districts)} Kandidaten verworfen (Lage/Groesse/Namensdopplung)")

    if len(districts) == 0:
        print("\n⚠️  Nach Filterung keine Stadtteile uebrig. Gruende:")
        print(f"     Ueberlappung <= 50%:    {int((dbg['overlap_frac'] <= 0.5).sum())}")
        print(f"     Flaeche < {MIN_AREA_KM2} km²:   "
              f"{int((dbg['area_km2'] < MIN_AREA_KM2).sum())}")
        print(f"     Flaeche >= halbe Stadt: {int((dbg['area_km2'] >= city_area * 0.5).sum())}")
        diagnose_osm_districts(CITY_NAME)
        HAS_DISTRICTS = False
    else:
        districts = resolve_nesting(districts, NESTING_STRATEGY, NESTING_THRESHOLD)
        if len(districts) < MIN_DISTRICTS_REQ:
            print(f"\n⚠️  Nur {len(districts)} Stadtteil(e) gefunden "
                  f"(Minimum: {MIN_DISTRICTS_REQ}). Eine stadtteilbezogene "
                  f"Auswertung waere nicht aussagekraeftig.")
            HAS_DISTRICTS = False

    if HAS_DISTRICTS:
        districts_gdf = gpd.GeoDataFrame(districts, geometry="geometry", crs="EPSG:4326")
        print(f"\n{len(districts_gdf)} Stadtteile final:")
        for _, row in districts_gdf.sort_values("name").iterrows():
            print(f"  {row['name']:32s} {row['area_km2']:6.2f} km²")

        total_area = districts_gdf["area_km2"].sum()
        coverage_pct = total_area / city_area * 100 if city_area > 0 else float("nan")
        print(f"\nSumme Stadtteilflaechen: {total_area:.1f} km² "
              f"({coverage_pct:.0f}% der Stadtflaeche)")
        if coverage_pct > 110:
            print("⚠️  Ueber 110% -- vermutlich noch Ueberlappungen. "
                  "NESTING_THRESHOLD in config.yaml senken (z.B. 0.5).")
        elif coverage_pct < 80:
            print("⚠️  Unter 80% -- Luecken zwischen den Polygonen. "
                  "Bilder dort bleiben unzugeordnet.")

if not HAS_DISTRICTS:
    print("=" * 62)
    print("KEINE STADTTEILE VERFUEGBAR")
    print("Uebersprungen werden: Metriken pro Stadtteil, Choroplethen-Karte.")
    print("Weiterhin ausgewertet: stadtweite Heatmap, Strassennetz, Gesamtmetriken.")
    print("=" * 62)


## 9. Abdeckung pro Stadtteil


In [ ]:
if not HAS_DISTRICTS:
    joined, metrics_df = None, None
    print("Uebersprungen: zu wenige Stadtteile gefunden.")
else:
    joined = (gpd.sjoin(images_gdf, districts_gdf[["name", "geometry"]],
                        how="left", predicate="within")
              .rename(columns={"name": "district"}))

    # Bilder exakt auf einer Stadtteilgrenze koennen bei restlichen
    # Ueberlappungen zwei Treffer bekommen -- erste Zuordnung gewinnt.
    n_dupes = int(joined["image_id"].duplicated().sum())
    if n_dupes:
        print(f"⚠️  {n_dupes} Bilder mehrfach zugeordnet (Restueberlappung) -- "
              f"behalte je erste Zuordnung")
        joined = joined.drop_duplicates(subset="image_id", keep="first").reset_index(drop=True)

    n_unassigned = int(joined["district"].isna().sum())
    pct_unassigned = n_unassigned / len(joined) * 100 if len(joined) else 0.0
    print(f"Ohne Stadtteil-Zuordnung: {n_unassigned:,} ({pct_unassigned:.1f}%)")
    if pct_unassigned > 20:
        print("⚠️  Ueber 20% unzugeordnet -- die Stadtteil-Polygone decken das "
              "Stadtgebiet nur lueckenhaft ab.")
    print()

    metrics_df = pd.DataFrame([
        {"Stadtteil":  r["name"],
         "Fläche km²": round(r["area_km2"], 2),
         "Bilder":     int((joined["district"] == r["name"]).sum()),
         "Bilder/km²": round((joined["district"] == r["name"]).sum() / r["area_km2"]),
         "Sequenzen":  joined.loc[joined["district"] == r["name"], "sequence_id"].nunique()}
        for _, r in districts_gdf.iterrows()
    ]).sort_values("Bilder/km²", ascending=False).reset_index(drop=True)

    zugeordnet = int(metrics_df["Bilder"].sum())
    median = metrics_df["Bilder/km²"].median()
    print(f"Zugeordnet: {zugeordnet:,} von {len(images_gdf):,} "
          f"({zugeordnet / len(images_gdf) * 100:.1f} %)")
    print(f"Median:     {median:,.0f} Bilder/km²\n")

    schwach = metrics_df[metrics_df["Bilder/km²"] < median * 0.5]
    if len(schwach):
        print(f"{len(schwach)} Stadtteile unter der Haelfte des Medians:")
        for _, r in schwach.iterrows():
            print(f"   {r['Stadtteil']:<32} {r['Bilder/km²']:>8,.0f}")

metrics_df


## 10. Choroplethenkarte


In [ ]:
if not HAS_DISTRICTS:
    print("Übersprungen: braucht Stadtteile.")
else:
    plot_gdf = districts_gdf.merge(metrics_df, left_on="name",
                                   right_on="Stadtteil", how="left")

    fig, ax = plt.subplots(figsize=(13, 11))
    plot_gdf.plot(column="Bilder/km²", cmap="YlOrRd", legend=True, ax=ax,
                  edgecolor="black", linewidth=0.5,
                  legend_kwds={"label": "Bilder pro km²", "shrink": 0.7},
                  missing_kwds={"color": "lightgrey", "label": "keine Daten"})

    for _, r in plot_gdf.iterrows():
        p = r.geometry.representative_point()      # liegt garantiert im Polygon
        ax.annotate(r["name"], xy=(p.x, p.y), fontsize=7, ha="center", va="center",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.65, ec="none"))

    gpd.GeoSeries([city_polygon], crs="EPSG:4326").boundary.plot(
        ax=ax, color="black", linewidth=1.5, linestyle="--")
    ax.set_aspect(1 / np.cos(np.radians(city_polygon.centroid.y)))
    ax.set_title(f"Mapillary-Abdeckung nach Stadtteil — {CITY_NAME.split(',')[0]}")
    plt.tight_layout(); plt.savefig(DATA_DIR / "choropleth_coverage.png"); plt.show()


## 11. Dichte-Heatmap

Geglättetes 2D-Histogramm statt Kerndichteschätzung: eine echte KDE müsste für
jeden Rasterpunkt über alle 300.000 Bilder summieren und läuft bei dieser Menge
praktisch nicht mehr. Das Ergebnis sieht gleich aus und rechnet in Sekunden.


In [ ]:
from scipy.ndimage import gaussian_filter

pad_x = (lon_max - lon_min) * 0.03
pad_y = (lat_max - lat_min) * 0.03
extent = [lon_min - pad_x, lon_max + pad_x, lat_min - pad_y, lat_max + pad_y]

density, _, _ = np.histogram2d(images_gdf["lon"], images_gdf["lat"],
                               bins=[420, 300],
                               range=[extent[:2], extent[2:]])
density = gaussian_filter(density, sigma=2.0)

fig, ax = plt.subplots(figsize=(13, 11))
# log1p, weil die Innenstadt sonst alles andere überstrahlt
im = ax.imshow(np.log1p(density).T, origin="lower", extent=extent,
               cmap="inferno", interpolation="bilinear", aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.7, label="Bilddichte (logarithmisch)")

gpd.GeoSeries([city_polygon], crs="EPSG:4326").boundary.plot(
    ax=ax, color="white", linewidth=1.5, linestyle="--")
ax.set_aspect(1 / np.cos(np.radians(city_polygon.centroid.y)))
ax.set_title(f"Bilddichte — {CITY_NAME.split(',')[0]} ({len(images_gdf):,} Bilder)")
plt.tight_layout(); plt.savefig(DATA_DIR / "heatmap_coverage.png"); plt.show()


## 12. Bericht


In [ ]:
per_seq = images_gdf["sequence_id"].value_counts()
captured = pd.to_datetime(images_gdf["captured_at"], unit="ms", errors="coerce")

print("=" * 58)
print(f"  Stadt                {CITY_NAME}")
print(f"  Fläche               {city_area:,.1f} km²")
print(f"  Bilder               {len(images_gdf):,}")
print(f"  Dichte               {len(images_gdf) / city_area:,.0f} /km²")
print(f"  Sequenzen            {per_seq.size:,}")
print(f"  Median je Sequenz    {per_seq.median():,.0f}   (längste: {per_seq.max():,})")
print(f"  Aufnahmezeitraum     {captured.min():%Y-%m} bis {captured.max():%Y-%m}")
if HAS_DISTRICTS:
    print(f"  Stadtteile           {len(districts_gdf)}")
print("=" * 58)


## Anhang — Achsenkalibrierung

Nur nötig, wenn sich die Dekodierbibliothek ändert oder die Punkte plötzlich
verschoben aussehen. Vergleicht beide Achsenrichtungen gegen Koordinaten aus
der Graph-API und meldet, welche stimmt. Braucht eine Referenzdatei mit den
Spalten `image_id`, `lon`, `lat`.


In [ ]:
REF_FILE = DATA_DIR / "images_raw.csv"      # bei Bedarf anpassen

if not REF_FILE.exists():
    print(f"Keine Referenzdatei ({REF_FILE}) — Kalibrierung übersprungen.")
else:
    ref = pd.read_csv(REF_FILE).set_index("image_id")[["lon", "lat"]]
    cx, cy = deg2tile((lon_min + lon_max) / 2, (lat_min + lat_max) / 2, ZOOM)
    session = make_session()

    for candidate in (False, True):
        Y_DOWN = candidate                              # noqa: F811
        pts = pd.DataFrame(load_tile(session, cx, cy))
        both = pts.set_index("image_id").join(ref, rsuffix="_ref", how="inner").dropna()
        if both.empty:
            print(f"  y_down={candidate}: keine gemeinsamen Bilder")
            continue
        dist = np.hypot(
            (both["lon"] - both["lon_ref"]) * 111_320 * np.cos(np.radians(both["lat"])),
            (both["lat"] - both["lat_ref"]) * 111_320)
        print(f"  y_down={candidate!s:<5} {len(both):>5} Treffer, "
              f"Medianabweichung {dist.median():>10,.1f} m")

    Y_DOWN = False       # gemessenes Ergebnis wiederherstellen
    print("\nDer kleinere Wert gewinnt. Liegen beide über 25 m, stimmt an der "
          "Dekodierung etwas nicht.")
